inferring：
- 情感推断：情感倾向分析（正面/负面）；识别情感类型（list）；识别愤怒
- 信息提取：提取关心的信息；信息提取与情感推断结合
- 主题推断：讨论的主题；是否包含某些主题

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

import openai

In [2]:
def get_completion(prompt):
    message = [{"role":"user", "content": prompt}]

    client = openai.OpenAI(
        base_url=os.getenv("OLLAMA_BASE_URL"),
        api_key="ollama",
    )

    response = client.chat.completions.create(
        messages=message,
        model=os.getenv("MODEL_NAME"),
        temperature=0.0,
    )
    return response.choices[0].message.content

In [3]:
lamp_review = """
我需要一盏漂亮的卧室灯，这款灯具有额外的储物功能，价格也不算太高。\
我很快就收到了它。在运输过程中，我们的灯绳断了，但是公司很乐意寄送了一个新的。\
几天后就收到了。这款灯很容易组装。我发现少了一个零件，于是联系了他们的客服，他们很快就给我寄来了缺失的零件！\
在我看来，Lumina 是一家非常关心顾客和产品的优秀公司！
"""

情感推断

In [4]:
### 情感倾向分析

prompt = f"""
以下用三个反引号分隔的产品评论的情感是什么？

评论文本：```{lamp_review}```
"""

response = get_completion(prompt)
print(response)

这段产品评论的情感是积极的。评论中提到了对产品的多个正面评价，包括其美观、额外功能、合理的价格以及良好的售后服务等。此外，评论者还特别赞扬了公司的服务态度和产品质量，认为Lumina是一家关心顾客和产品的优秀公司。整体来看，这是一个非常满意的客户反馈。


In [5]:
### 要求回答简洁

prompt = f"""
以下用三个反引号分隔的产品评论的情感是什么？

用一个词回答：“正面”或“负面”

评论文本：```{lamp_review}```
"""

response = get_completion(prompt)
print(response)


正面


In [6]:
### 识别情感类型

prompt = f"""
识别以下用三个反引号分隔的评论中作者表达的情感。\
包含不超过5项的列表。\
答案格式为用逗号分隔的列表。

评论文本：```{lamp_review}```
"""

response = get_completion(prompt)
print(response)

满意, 感激, 高兴, 信任


In [9]:
### 识别愤怒

prompt = f"""
以下用三个反引号分隔的评论作者是否表达了愤怒？

用“是”或“否”回答。

评论文本: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)

否


信息提取

In [10]:
### 从评论中关心的信息，并且要求输出以JSON格式

prompt = f"""
从用三个反引号分隔的评论文本中识别以下项目：
- 评论者购买的物品
- 制造该物品的公司

格式化为以 “物品” 和 “品牌” 为键的 JSON 对象。
如果信息不存在，请使用 “未知” 作为值。
尽可能简短。

评论文本: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)

```json
{
  "物品": "卧室灯",
  "品牌": "Lumina"
}
```


In [11]:
### 信息提取 + 情感分析 结合

prompt = f"""
从以下用三个反引号分隔的评论文本中识别以下项目：
- 情绪<“正面”或“负面”>
- 评论者是否表达了愤怒？<"True"/"FALSE">
- 评论者购买的物品
- 制造该物品的公司

格式化为 JSON 对象，以 “情感倾向”、“是否生气”、“物品类型” 和 “品牌” 作为键。
如果信息不存在，请使用 “未知” 作为值。
尽可能简短。
将 “是否生气” 值格式化为布尔值。

评论文本: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)

```json
{
  "情感倾向": "正面",
  "是否生气": false,
  "物品类型": "卧室灯",
  "品牌": "Lumina"
}
```


主题推断

In [12]:
story = """
在政府最近进行的一项调查中，要求公共部门的员工对他们所在部门的满意度进行评分。
调查结果显示，NASA 是最受欢迎的部门，满意度为 95％。

一位 NASA 员工 John Smith 对这一发现发表了评论，他表示：
“我对 NASA 排名第一并不感到惊讶。这是一个与了不起的人们和令人难以置信的机会共事的好地方。我为成为这样一个创新组织的一员感到自豪。”

NASA 的管理团队也对这一结果表示欢迎，主管 Tom Johnson 表示：
“我们很高兴听到我们的员工对 NASA 的工作感到满意。
我们拥有一支才华横溢、忠诚敬业的团队，他们为实现我们的目标不懈努力，看到他们的辛勤工作得到回报是太棒了。”

调查还显示，社会保障管理局的满意度最低，只有 45％的员工表示他们对工作满意。
政府承诺解决调查中员工提出的问题，并努力提高所有部门的工作满意度。
"""

In [13]:
### 推断主题内容

prompt = f"""
确定以下用三个反引号分隔的给定文本中讨论的五个主题。

每个主题用1-2个词概括。

请输出一个可解析的Python列表，每个元素是一个主题的字符串，用逗号分隔。

给定文本: ```{story}```
"""
response = get_completion(prompt)
print(response)

```python
["NASA", "满意度调查", "员工评论", "管理团队反应", "问题解决"]
```


In [15]:
### 确定文本中是否包含以下主题

topic_list = ["nasa", "当地政府", "工程", "职工满意度", "联邦政府"]

prompt = f"""
判断下面主题列表中的每一项是否是给定文本中的一个话题，

以列表的形式给出答案，每个元素是一个Json对象，键为对应主题，值为对应的 0 或 1。

主题列表：{topic_list}

给定文本: ```{story}```
"""
response = get_completion(prompt)
print(response)

```json
[
    {"nasa": 1},
    {"当地政府": 0},
    {"工程": 0},
    {"职工满意度": 1},
    {"联邦政府": 0}
]
```
